In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from io import StringIO
import pandas as pd

def scrape_sp500():
    options = webdriver.ChromeOptions()
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=options)

    try:
        url = (
            "https://finance.yahoo.com/quote/%5EGSPC/history/"
            "?period1=-1325583000&period2=1790689249&frequency=1d"
        )
        driver.get(url)

        table = WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "table"))
        )

        table_html = table.get_attribute("outerHTML")

        df = pd.read_html(StringIO(table_html))[0]

        df = df[
            ~df.astype(str).apply(
                lambda row: row.str.contains(r"\*", regex=True).any(),
                axis=1
            )
        ]

        df.columns = [
            "Date",
            "Open",
            "High",
            "Low",
            "Close",
            "Adj Close",
            "Volume"
        ]

        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"])
        df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")

        csv_filename = "sp500_yahoo.csv"
        df.to_csv(csv_filename, index=False, encoding="utf-8")

        tsv_filename = "sp500_yahoo.tsv"
        df.to_csv(tsv_filename, sep="\t", index=False, encoding="utf-8")

        json_filename = "sp500_yahoo.json"
        df.to_json(json_filename, orient="records", indent=4, force_ascii=False)

        return df

    except Exception:
        return None

    finally:
        driver.quit()


df = scrape_sp500()